# Bee Haven – Gold Data Processing

This notebook combines the processed Bee Haven sensor and weather data from the
Gold processing layer into a unified Gold dataset.

### Processing steps

1. Read processed Parquet files from `gold/processing`
2. Combine sensor and weather data by location and timestamp
3. Standardize the dataset structure
4. Create a unified Gold dataset
5. Create a daily aggregated dataset
6. Write the Gold datasets to Azure Data Lake Storage

### Data sources

- Flow
- Humidity
- Temperature
- Weight
- Weather

### Azure

The notebook is designed to run in an Azure Synapse Spark environment.

------------

#### Azure Configuration

In [ ]:
import pandas as pd
import fsspec

# Azure Data Lake Storage configuration
# Replace these values with your own Azure Storage account details.

STORAGE_ACCOUNT = "YOUR_STORAGE_ACCOUNT"
CONTAINER = "YOUR_CONTAINER_NAME"

root_dir = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

source = root_dir + "gold/processing/"
sink = root_dir + "gold/"

storage_options = {
    "account_name": STORAGE_ACCOUNT,
    "account_key": "YOUR_ACCOUNT_KEY"
}

####  Connection

In [ ]:
# Connect to Azure Data Lake Storage
# 'abfss': azure blob file system
fs = fsspec.filesystem(
    "abfss",
    account_name=storage_options["account_name"]
)

####  Location DataFrames

In [ ]:
# Initialize an empty DataFrame for each Bee Haven location
location_dfs = {
    "schwartau": pd.DataFrame(),
    "wurzburg": pd.DataFrame()
}

####  Read and Merge

In [ ]:
# List files in the Gold processing directory
file_list = fs.ls(source)

In [ ]:
# Read and merge processed files by location
for file_path in file_list:
    file_name = file_path.rsplit("/", maxsplit=1)[-1]

    # Skip hidden files
    if file_name.startswith("."):
        continue

    print(f"Processing {file_name}")

    df = pd.read_parquet(file_path, filesystem=fs)
    location = file_name.split("_")[0]

    if location not in location_dfs:
        print(f"Skipping file with unknown location: {file_name}")
        continue

    # Rename hive-specific temperature and humidity measurements
    if df.shape[1] == 2:
        if "temperature" in df.columns:
            df.columns = ["timestamp", "temperature_hive"]
        elif "humidity" in df.columns:
            df.columns = ["timestamp", "humidity_hive"]

    if location_dfs[location].empty:
        location_dfs[location] = df.copy()
    else:
        location_dfs[location] = location_dfs[location].merge(
            df,
            on="timestamp",
            how="outer"
        )

### Prepare Location Data

After merging the processed files by timestamp, location information is added to each dataset and the records are sorted chronologically.

In [ ]:
# Add location information and sort by timestamp
for location, df in location_dfs.items():
    # skip locations with no DataFrames
    if df.empty:
        continue
    # add location
    df["location"] = location
    df["location"] = df["location"].astype("category")
    # sort by timestamps
    location_dfs[location] = df.sort_values("timestamp")

### Create Gold Dataset

The location-specific datasets are combined into a single Gold dataset with a
consistent column order.

In [ ]:
gold_columns = [
    "timestamp",
    "location",
    "flow_out",
    "flow_in",
    "temperature_hive",
    "humidity_hive",
    "weight",
    "precipitation",
    "pressure_msl",
    "sunshine",
    "temperature",
    "wind_direction",
    "wind_speed",
    "cloud_cover",
    "dew_point",
    "relative_humidity",
    "wind_gust_direction",
    "wind_gust_speed",
    "solar",
]

gold = pd.concat(
    [location_dfs["schwartau"], location_dfs["wurzburg"]],
    ignore_index=True
)[gold_columns]

gold.head()

### Save Gold Dataset

The merged Gold dataset is written to Azure Data Lake Storage in Parquet format.

In [ ]:
write_name = (
    sink
    + f"hivedata__{pd.Timestamp.now().strftime('%Y-%m-%dT%Hh%Mm%Ss')}.parquet"
)

with fs.open(write_name, "wb") as file:
    gold.to_parquet(file, index=False)

### Create Daily Gold Dataset

The Gold dataset is aggregated to daily frequency for each location to create
a cleaned dataset suitable for downstream analysis.

In [ ]:
gold_clean = (
    gold
    .set_index("timestamp")
    .groupby("location")
    .resample("D")
    .mean()
    .reset_index()
)

gold_clean.describe()

### Save Cleaned Gold Dataset

The daily aggregated dataset is written to Azure Data Lake Storage as a separate
Parquet file for downstream analysis.

In [ ]:
write_name = (
    sink
    + f"hivedata_clean__{pd.Timestamp.now().strftime('%Y-%m-%dT%Hh%Mm%Ss')}.parquet"
)

with fs.open(write_name, "wb") as file:
    gold_clean.to_parquet(file, index=False)

In [ ]:
# Release the Synapse Spark session
mssparkutils.session.stop()